# Atari Mnih smoke test

Run a short Atari training job from the notebook kernel, then verify its TensorBoard scalars. The default configuration uses the `arxiv2013` preset on Breakout, CPU execution, and small in-loop evaluation and held-out intervals. Edit the configuration cell to try another preset or environment.

Run the cells from top to bottom. Training runs directly through Python's `runpy`; no shell command is used.

In [ ]:
from pathlib import Path
import runpy
import sys
import time

import torch

# Editable smoke-test settings.
preset = "arxiv2013"
env_id = "BreakoutNoFrameskip-v4"
total_timesteps = 3000
learning_starts = 500
eval_steps = 500
eval_frequency = 1000
heldout_frequency = 1000
buffer_size = 10_000
batch_size = 32
seed = 1

torch.set_num_threads(1)
argv = [
    "atari_mnih.py",
    "--preset", preset,
    "--env-id", env_id,
    "--total-timesteps", str(total_timesteps),
    "--learning-starts", str(learning_starts),
    "--eval-steps", str(eval_steps),
    "--eval-frequency", str(eval_frequency),
    "--heldout-frequency", str(heldout_frequency),
    "--buffer-size", str(buffer_size),
    "--batch-size", str(batch_size),
    "--seed", str(seed),
    "--no-cuda",
]

original_argv = sys.argv
run_started_at = time.time()
try:
    sys.argv = argv
    runpy.run_module("atari_mnih", run_name="__main__")
finally:
    sys.argv = original_argv

print("Smoke run completed.")

## Verify the run

This cell loads the TensorBoard event file created by the run above and checks that in-loop evaluation, held-out Q logging, and finite scalar values are present.

In [ ]:
import math

from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

run_dirs = [
    path
    for path in Path("runs").glob(f"{env_id}__atari_mnih__{seed}__*")
    if path.is_dir() and path.stat().st_mtime >= run_started_at - 2
]
assert run_dirs, "No TensorBoard run directory was created by the smoke run."
run_dir = max(run_dirs, key=lambda path: path.stat().st_mtime)

event_files = sorted(run_dir.glob("events.out.tfevents.*"))
assert event_files, f"No TensorBoard event file found in {run_dir}."
event_accumulator = EventAccumulator(str(run_dir), size_guidance={"scalars": 0})
event_accumulator.Reload()
scalar_tags = event_accumulator.Tags().get("scalars", [])

required_tags = {
    "eval/mean_return",
    "eval/max_return",
    "eval/num_episodes",
    "eval/partial",
    "charts/heldout_avg_max_q",
}
missing_tags = required_tags.difference(scalar_tags)
assert not missing_tags, f"Missing expected TensorBoard scalars: {sorted(missing_tags)}"

all_scalars = {
    tag: event_accumulator.Scalars(tag)
    for tag in scalar_tags
}
assert all(
    math.isfinite(event.value)
    for events in all_scalars.values()
    for event in events
), "The run logged a non-finite scalar."

for tag in sorted(required_tags):
    events = all_scalars[tag]
    print(f"{tag}: steps={[event.step for event in events]}, values={[event.value for event in events]}")

print(f"Verified {len(scalar_tags)} scalar tags in {run_dir}")